# Environment

In [72]:
%pip install tqdm openai dotenv
%pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [73]:
from tqdm import tqdm
import pandas as pd
import json
import os
import time
from openai import AzureOpenAI
from dotenv import load_dotenv

In [74]:
import csv, math
from collections import defaultdict, Counter
from pprint import pprint


# load_azure_openai_client?

In [75]:

# Load environment variables from .env file
def load_azure_openai_client():
    load_dotenv()
    return AzureOpenAI(
        api_version=os.getenv('api_version'),
        azure_endpoint=os.getenv('api_endpoint'),
        api_key=os.getenv('api_key'),
    )


# Cloud 

### Load Dataset

In [4]:
import pandas as pd
from pathlib import Path

cloud = pd.read_excel("../../data/splits/cloud/cloud_test.xlsx")
cloud_test = pd.read_excel("../../data/splits/cloud/cloud_test.xlsx")

print("cloud:", cloud.shape)
print("cloud_test:", cloud_test.shape)


cloud: (235, 15)
cloud_test: (1568, 13)


In [5]:
cloud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   index            235 non-null    int64 
 1   link             235 non-null    object
 2   question         235 non-null    object
 3   vlm_zero         235 non-null    object
 4   vlm_cot          235 non-null    object
 5   llm_text_zero    235 non-null    object
 6   llm_text_cot     235 non-null    object
 7   llm_ocr_zero     235 non-null    object
 8   llm_ocr_cot      235 non-null    object
 9   sft_zero         225 non-null    object
 10  dpo_zero         235 non-null    object
 11  ref_answer       235 non-null    object
 12  ref_dpo_verdict  235 non-null    object
 13  gpt_verdict      235 non-null    object
 14  Select           235 non-null    object
dtypes: int64(1), object(14)
memory usage: 27.7+ KB


In [6]:
cloud['ref_answer'] = [x for x in cloud['ref_answer'] if x.strip()]


### Filter 

In [7]:
#actual index mapping with main data

link_to_idx = (
    cloud_test
        .reset_index()          
        .set_index('link')['index']   # use 'link' as index, and keep the original index as values
)

# Map device['link'] through that mapping
cloud['index'] = cloud['link'].map(link_to_idx)

In [8]:
cloud.loc[3]

index                                                            330
link               https://stackoverflow.com/questions/37893059/s...
question           You are an expert cloud assistant. Provide a c...
vlm_zero           The issue you're experiencing is due to the fa...
vlm_cot            To resolve the issue, follow these steps:\n\n1...
llm_text_zero      The issue is that the IAM user you created doe...
llm_text_cot       The issue is that the IAM user you created doe...
llm_ocr_zero       The issue here seems to be related to permissi...
llm_ocr_cot        The issue here seems to be related to permissi...
sft_zero           The issue here was due to me using an IAM User...
dpo_zero           The issue here seems unrelated to the actual p...
ref_answer         You skipped step 3 in the Getting Started guid...
ref_dpo_verdict                                                  DPO
gpt_verdict                                            llm_text_zero
Select                            

In [9]:
# images from testset
cloud['images'] = cloud['index'].apply(
    lambda idx: cloud_test.loc[idx, 'images']
)

In [10]:
cloud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   index            235 non-null    int64 
 1   link             235 non-null    object
 2   question         235 non-null    object
 3   vlm_zero         235 non-null    object
 4   vlm_cot          235 non-null    object
 5   llm_text_zero    235 non-null    object
 6   llm_text_cot     235 non-null    object
 7   llm_ocr_zero     235 non-null    object
 8   llm_ocr_cot      235 non-null    object
 9   sft_zero         225 non-null    object
 10  dpo_zero         235 non-null    object
 11  ref_answer       235 non-null    object
 12  ref_dpo_verdict  235 non-null    object
 13  gpt_verdict      235 non-null    object
 14  Select           235 non-null    object
 15  images           235 non-null    object
dtypes: int64(1), object(15)
memory usage: 29.5+ KB


In [11]:
#cloud['sft_zero'] = cloud['sft_zero'].fillna('No answer')

In [12]:
cloud.head(2)

,index,link,question,vlm_zero,vlm_cot,llm_text_zero,llm_text_cot,llm_ocr_zero,llm_ocr_cot,sft_zero,dpo_zero,ref_answer,ref_dpo_verdict,gpt_verdict,Select,images
0,673,https://stackoverflow.com/questions/58656602/i...,You are an expert cloud assistant. Provide a c...,To get all documents in a subcollection in Fir...,To get all documents in a subcollection in Fir...,To query all documents in a subcollection in F...,To query all documents in a subcollection in F...,The issue lies in your `db.collection().doc()....,The issue lies in your `db.collection().doc()....,The issue here is that `db.collection().doc()`...,The issue lies in your use of `db.collection()...,"To summarize, get() method has to be called to...",DPO,vlm_zero,yes,['https://i.sstatic.net/3Svyy.png']
1,1476,https://stackoverflow.com/questions/54523755/a...,You are an expert cloud assistant. Provide a c...,"To resolve the issue, you need to change the D...",To resolve the issue of not seeing the updated...,To reload the container and apply the added en...,To restart the container and apply the new env...,The solution is to change the Docker image and...,To resolve the container restart issue for the...,NaN,The solution is to change the Docker image and...,What you need to do is change the image from g...,DPO,**dpo_zero**,maybe,"['https://i.sstatic.net/U6bsL.png', 'https://i..."


In [13]:
print(cloud['question'][0])

You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Tables:
Table 0:
+------------------------------+----------------------------------------+---+
| users                        | q1Cj68vKicgdXt9flxMQkC6qgyf1           | ? |
|                              | ...                                    |   |
+------------------------------+----------------------------------------+---+
| + Add document               | + Start collection                     | + |
+------------------------------+----------------------------------------+---+
| LKB3YgolzsNCyQVNfc9ToQrdEq53 | favorites                              |   |
|                              | >                                      |   |
+------------------------------+----------------------------------------+---+
| q1Cj68vKicgdXt9fIxMQkC6qgyf1 |                    

### Extract Title + Body 

In [14]:
import re

def extract_ocr_and_question(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Grab OCR Text block (between "OCR Text:" and "Question:")
    ocr_pattern = re.compile(
        r"OCR Text:\s*(?P<ocr>.*?)(?=\n\s*Question:)",
        re.DOTALL
    )
    ocr_match = ocr_pattern.search(text)

    # Grab Question title/body (like before)
    question_pattern = re.compile(
        r"Question:\s*"
        r"title:\s*(?P<title>.+?)\s*\n"
        r"body:\s*(?P<body>.*?)(?=\n\s*Answer:)",
        re.DOTALL
    )
    q_match = question_pattern.search(text)

    if not q_match:
        return ""

    title = q_match.group("title").strip()
    body = q_match.group("body").strip()

    parts = []

    if ocr_match:
        ocr_text = ocr_match.group("ocr").strip()
        parts.append(f"OCR Text:\n{ocr_text}")

    parts.append(f"Question:\ntitle: {title}\nbody: {body}")

    return "\n\n".join(parts)

# usage
cloud["raw_question"] = cloud["question"].apply(extract_ocr_and_question)

In [15]:
print(cloud['raw_question'][3])

OCR Text:
img0:
Non-table text:
 Service Role
 Select an existing service role that grants AWS CodeDeploy access to the instances.
                                                                    None
                                    Service Role ARN*

Question:
title: Service Role ARN blank in CodeDeploy
body: I am trying to setup the CodeDeploy service with one of my ec2 instances. On the form to do this, everything works properly until I reach the "Service Role" section. 

This field does not display any possible ARNs.

This is what I see:



I created an IAM user and gave them Admin privileges and pasted their ARN into this field and got the error message


  Cannot assume role provided.


Does anyone know the reason this is happening?


In [16]:
print(cloud['dpo_zero'][3])

The issue here seems unrelated to the actual problem at hand but rather a symptom of it. The correct solution would be to select an existing service role or create a new one specifically for CodeDeploy instead of using an arbitrary IAM user's ARN. This will ensure proper permissions and avoid errors like 'Cannot assume role provided'. To resolve your current situation, try selecting an existing service role from the dropdown menu available under the 'Select an existing service role' option. If no roles appear, verify that there are indeed service roles associated with your account by checking the IAM dashboard. If still unable to find suitable options, consider creating a custom service role tailored for CodeDeploy operations.


In [17]:
print(cloud['llm_text_zero'][3])

The issue is that the IAM user you created does not have the necessary permissions to assume the role. To resolve this, you need to create an IAM role with the necessary permissions and then use that role's ARN in the CodeDeploy service.


### Prompt

#### Orignal Prompt from Paper 

In [18]:
'''
SYSTEM_JUDGE_PROMPT = """
Please act as an impartial judge and evaluate the quality of the responses provided by two AI assistants to
the user prompt displayed below. You will be given assistant A’s answer and assistant B’s answer. Your
job is to evaluate which assistant’s answer is better.

Begin your evaluation by generating your own answer to the prompt. You must provide your answers before
judging any answers.

When evaluating the assistants’ answers, compare both assistants’ answers with your answer. You must
identify and correct any mistakes or inaccurate information.

Then consider if the assistant’s answers are helpful, relevant, and concise. Helpful means the answer
correctly responds to the prompt or follows the instructions. Note when user prompt has any ambiguity or
more than one interpretation, it is more helpful and appropriate to ask for clarifications or more information
from the user than providing an answer based on assumptions. Relevant means all parts of the response
closely connect or are appropriate to what is being asked. Concise means the response is clear and not
verbose or excessive.

Then consider the creativity and novelty of the assistant’s answers when needed. Finally, identify any
missing important information in the assistants’ answers that would be beneficial to include when responding
to the user prompt.
After providing your explanation, you must output only one of the following choices as your final verdict
with a label:
1. Assistant A is significantly better: [[A>>B]]
2. Assistant A is slightly better: [[A>B]]
3. Tie, relatively the same: [[A=B]]
4. Assistant B is slightly better: [[B>A]]
5. Assistant B is significantly better: [[B>>A]

Example output: "My final verdict is tie: [[A=B]]"
"""

'''

'\nSYSTEM_JUDGE_PROMPT = """\nPlease act as an impartial judge and evaluate the quality of the responses provided by two AI assistants to\nthe user prompt displayed below. You will be given assistant A’s answer and assistant B’s answer. Your\njob is to evaluate which assistant’s answer is better.\n\nBegin your evaluation by generating your own answer to the prompt. You must provide your answers before\njudging any answers.\n\nWhen evaluating the assistants’ answers, compare both assistants’ answers with your answer. You must\nidentify and correct any mistakes or inaccurate information.\n\nThen consider if the assistant’s answers are helpful, relevant, and concise. Helpful means the answer\ncorrectly responds to the prompt or follows the instructions. Note when user prompt has any ambiguity or\nmore than one interpretation, it is more helpful and appropriate to ask for clarifications or more information\nfrom the user than providing an answer based on assumptions. Relevant means all par

#### Tuned Prompt

In [19]:
SYSTEM_JUDGE_PROMPT = """
Please act as an impartial judge and evaluate the quality of the responses provided by two AI assistants to
the user prompt displayed below. You will be given a reference answer, assistant A’s answer, and assistant B’s answer. 
Your job is to evaluate which assistant’s answer is better.

Begin your evaluation by reviewing the reference answer to the prompt. You must use this reference answer, including
“No Answer” cases (which mean the prompt is unanswerable), as your correctness baseline before judging any assistant answer.

When evaluating the assistants’ answers, compare both assistants’ answers with the reference answer. You must
identify and correct any mistakes or inaccurate information. Your evaluation must remain unbiased: do not favor answers
based on verbosity, structure, formatting, or length. An answer with incorrect or irrelevant detail must be penalized
even if it appears more complete.

Then consider if the assistant’s answers are helpful, relevant, and concise. Helpful means the answer
correctly responds to the prompt or follows the instructions. Note when user prompt has any ambiguity or
more than one interpretation, it is more helpful and appropriate to ask for clarifications or more information
from the user than providing an answer based on assumptions. Relevant means all parts of the response closely
connect or are appropriate to what is being asked. Concise means the response is clear and not verbose or excessive.

Then consider the creativity and novelty of the assistant’s answers when needed. Finally, identify any
missing important information in the assistants’ answers that would be beneficial to include when responding
to the user prompt.

After providing your explanation, you must output only one of the following choices as your final verdict
with a label:
1. Assistant A is significantly better: [[A>>B]]
2. Assistant A is slightly better: [[A>B]]
3. Tie, relatively the same or both fail: [[A=B]]
4. Assistant B is slightly better: [[B>A]]
5. Assistant B is significantly better: [[B>>A]]

Example output: "My final verdict is tie: [[A=B]]"
"""

#### LLM Judge Setup

In [20]:
cloud_llm_judge = """
[User Question]
{question}
[The Start of Reference Answer]
{answer_ref}
[The End of Reference Answer]
[The Start of Assistant A’s Answer]
{answer_a}
[The End of Assistant A’s Answer]
[The Start of Assistant B’s Answer]
{answer_b}
[The End of Assistant B’s Answer]
"""

In [21]:
'''
cloud_llm_judge = """
[User Question]
{question}
[The Start of Assistant A’s Answer]
{answer_a}
[The End of Assistant A’s Answer]
[The Start of Assistant B’s Answer]
{answer_b}
[The End of Assistant B’s Answer]
"""
'''

'\ncloud_llm_judge = """\n[User Question]\n{question}\n[The Start of Assistant A’s Answer]\n{answer_a}\n[The End of Assistant A’s Answer]\n[The Start of Assistant B’s Answer]\n{answer_b}\n[The End of Assistant B’s Answer]\n"""\n'

In [22]:
print(cloud_llm_judge)


[User Question]
{question}
[The Start of Reference Answer]
{answer_ref}
[The End of Reference Answer]
[The Start of Assistant A’s Answer]
{answer_a}
[The End of Assistant A’s Answer]
[The Start of Assistant B’s Answer]
{answer_b}
[The End of Assistant B’s Answer]



#### Ranndomized Setup 

In [23]:

# --- Set 1 ---
A = "dpo_zero"
B_list_1 = ['vlm_cot']
#B_list_1=["vlm_zero"]
for B in B_list_1:
    col_name = f"{A}_vs_{B}"
    cloud[col_name] = cloud.apply(
        lambda row: cloud_llm_judge.format(
            
            question=row["raw_question"],
            answer_ref=row['ref_answer'],
            answer_a=row[A],
            answer_b=row[B]
        ),
        axis=1
    )

# --- Set 2 ---
A_list_2=['llm_text_zero']
B = "dpo_zero"

for A in A_list_2:
    col_name = f"{A}_vs_{B}"
    cloud[col_name] = cloud.apply(
        lambda row: cloud_llm_judge.format(
            question=row["raw_question"],
            answer_ref=row['ref_answer'],
            answer_a=row[A],
            answer_b=row[B]
        ),
        axis=1
    )

# the generated columns
cloud[[c for c in cloud.columns if "_vs_" in c]].head()

,dpo_zero_vs_vlm_cot,llm_text_zero_vs_dpo_zero
0,\n[User Question]\nOCR Text:\nimg0:\nTables:\n...,\n[User Question]\nOCR Text:\nimg0:\nTables:\n...
1,\n[User Question]\nOCR Text:\nimg0:\nNon-table...,\n[User Question]\nOCR Text:\nimg0:\nNon-table...
2,\n[User Question]\nOCR Text:\nimg0:\nTables:\n...,\n[User Question]\nOCR Text:\nimg0:\nTables:\n...
3,\n[User Question]\nOCR Text:\nimg0:\nNon-table...,\n[User Question]\nOCR Text:\nimg0:\nNon-table...
4,\n[User Question]\nOCR Text:\nimg0:\nNon-table...,\n[User Question]\nOCR Text:\nimg0:\nNon-table...


# Setup Messages

In [24]:
import ast


def setup_messages(dataset, prompt_column="", image_column="images"):
    conversations = []

    for _, row in dataset.iterrows():
        imgs = row.get(image_column, [])
        if isinstance(imgs, str):
            try:
                imgs = ast.literal_eval(imgs) 
            except Exception:
                imgs = []

        user_content = []

        #for img in imgs or []:
         #   if img:
         #       user_content.append({
         #           "type": "image_url",
         #           "image_url": {"url": img}
         #      })

        # Then the text prompt (your dpo_vs_sft_prompt)
        user_content.append({
            "type": "text",
            "text": row.get(prompt_column, "")
        })

        conversations.append([
            {"role": "system", "content": SYSTEM_JUDGE_PROMPT},
            {"role": "user", "content": user_content},
        ])

    return conversations


In [25]:
# List all the pairwise prompt columns you created
prompt_columns = [
    #'dpo_zero_vs_vlm_cot',
    #'dpo_zero_vs_llm_ocr_zero',
    #'llm_text_cot_vs_dpo_zero',
    #'dpo_zero_vs_llm_text_zero',
    #'vlm_zero_vs_dpo_zero'
   #'sft_zero_vs_dpo_zero',
   #'dpo_zero_vs_llm_text_cot',

   # orgina_paper:
   'dpo_zero_vs_vlm_cot',
   'llm_text_zero_vs_dpo_zero'
    
]

# Generate the corresponding _prompt_img_messages columns
for prompt_col in prompt_columns:
    msg_col = f"{prompt_col}_prompt_img_messages"
    cloud[msg_col] = setup_messages(cloud, prompt_column=prompt_col, image_column="images")

# Preview one
cloud[[c for c in cloud.columns if c.endswith("_prompt_img_messages")]].head(1)

,dpo_zero_vs_vlm_cot_prompt_img_messages,llm_text_zero_vs_dpo_zero_prompt_img_messages
0,"[{'role': 'system', 'content': ' Please act as...","[{'role': 'system', 'content': ' Please act as..."


In [26]:
cloud.columns

Index(['index', 'link', 'question', 'vlm_zero', 'vlm_cot', 'llm_text_zero',
       'llm_text_cot', 'llm_ocr_zero', 'llm_ocr_cot', 'sft_zero', 'dpo_zero',
       'ref_answer', 'ref_dpo_verdict', 'gpt_verdict', 'Select', 'images',
       'raw_question', 'dpo_zero_vs_vlm_cot', 'llm_text_zero_vs_dpo_zero',
       'dpo_zero_vs_vlm_cot_prompt_img_messages',
       'llm_text_zero_vs_dpo_zero_prompt_img_messages'],
      dtype='object')

## Extract Final verdict

In [27]:
import re

def extract_verdict(text):
    """Extract final verdict according to the prompt:
       [[A>>B]], [[A>B]], [[A=B]], [[B>A]], [[B>>A]]
    """
    if not isinstance(text, str):
        return None

    t = text.upper()

    # Pattern matches:
    # [[A>>B]], [[A>B]], [[A=B]], [[B>A]], [[B>>A]]
    pattern = r"\[\[\s*(A|B)\s*(>>|>|=)\s*(A|B)\s*\]\]"

    matches = re.findall(pattern, t)
    if matches:
        # Matches return tuples like ('A','>>','B')
        a, op, b = matches[-1]
        return f"[[{a}{op}{b}]]"

    # Fallback: phrases like "My final verdict is A>B"
    fallback_pattern = r"FINAL\s+VERDICT\s+IS\s+(A|B)\s*(>>|>|=)\s*(A|B)"
    m2 = re.search(fallback_pattern, t)
    if m2:
        a, op, b = m2.groups()
        return f"[[{a}{op}{b}]]"

    return None


# Inference

In [28]:
def inference(prompt, model_deployment_name="gpt-4o"):
    client = load_azure_openai_client()

    if isinstance(prompt, list):
        messages = prompt
    else:
        messages = [
            {"role": "user", "content": str(prompt)}
        ]

    response = client.chat.completions.create(
        model=model_deployment_name,
        messages=messages,
        seed=42,
    )
    return response.choices[0].message.content


In [29]:
import json
from tqdm import tqdm

def generate_responses(
    df,
    prompt_column: str,          # column with prompt/messages 
    output_column: str,          # column to store model output 
    output_jsonl: str,           # path to JSONL output file
    inference_fn,                # callable inference function
    opponent: str = "",          # e.g., "sft_zero"
    dpo_A: bool = False,  # whether llm_ocr_zero is Assistant A
    verdict_column: str = None,  
):
    # Default verdict column name
    if verdict_column is None:
        verdict_column = f"{output_column}_verdict"

    responses, verdicts = [], []

    # Open file once for appending results
    with open(output_jsonl, "a", encoding="utf-8") as f:
        for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating responses"):
            qid = int(row["index"]) if "index" in df.columns else int(idx)
            prompt_data = row.get(prompt_column, None)

            if prompt_data is None:
                print(f"[Skip] Missing prompt at row {idx}")
                responses.append(None)
                verdicts.append(None)
                continue

            try:
                # Run inference (works for text or message lists)
                response = inference_fn(prompt_data)
            except Exception as e:
                print(f"[Error] Row {idx} (qid={qid}): {e}")
                response = None

            # Extract structured verdict (your existing helper)
            verdict = extract_verdict(response) if response else None

            responses.append(response)
            verdicts.append(verdict)

            # JSONL record for evaluation logging
            record = {
                "index": qid,
                "prompt_column": prompt_column,
                "prompt": prompt_data,
                output_column: response,
                verdict_column: verdict,
                "opponent": opponent,
                "dpo_A": dpo_A, 
            }

            # Write in JSONL
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            # Print a sample for debugging (only first row)
            if idx == 0:
                print("\n--- Sample Prompt ---")
                if isinstance(prompt_data, list):
                    print(json.dumps(prompt_data[1]["content"], indent=2, ensure_ascii=False))
                else:
                    print(prompt_data)
                print("\n--- Sample Response ---\n", response)
                print("\n--- Sample Verdict ---\n", verdict)

    # Save results into DataFrame
    df[output_column] = responses
    df[verdict_column] = verdicts
    return df


In [30]:
cloud.head(1)

,index,link,question,vlm_zero,vlm_cot,llm_text_zero,llm_text_cot,llm_ocr_zero,llm_ocr_cot,sft_zero,...,ref_answer,ref_dpo_verdict,gpt_verdict,Select,images,raw_question,dpo_zero_vs_vlm_cot,llm_text_zero_vs_dpo_zero,dpo_zero_vs_vlm_cot_prompt_img_messages,llm_text_zero_vs_dpo_zero_prompt_img_messages
0,673,https://stackoverflow.com/questions/58656602/i...,You are an expert cloud assistant. Provide a c...,To get all documents in a subcollection in Fir...,To get all documents in a subcollection in Fir...,To query all documents in a subcollection in F...,To query all documents in a subcollection in F...,The issue lies in your `db.collection().doc()....,The issue lies in your `db.collection().doc()....,The issue here is that `db.collection().doc()`...,...,"To summarize, get() method has to be called to...",DPO,vlm_zero,yes,['https://i.sstatic.net/3Svyy.png'],OCR Text:\nimg0:\nTables:\nTable 0:\n+--------...,\n[User Question]\nOCR Text:\nimg0:\nTables:\n...,\n[User Question]\nOCR Text:\nimg0:\nTables:\n...,"[{'role': 'system', 'content': ' Please act as...","[{'role': 'system', 'content': ' Please act as..."


In [ ]:
from pathlib import Path
import pandas as pd

# Define where to save all JSONL outputs
OUTPUT_DIR = Path("../Evaluations/cloud")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for col in cloud.columns:
    if col.endswith("_prompt_img_messages"):
        base_name = col.replace("_prompt_img_messages", "")
        output_col = f"{base_name}_response"
        verdict_col = f"{output_col}_verdict"
        output_jsonl = OUTPUT_DIR / f"{base_name}_llm_judge.jsonl"

        # Determine model positions (A vs B)
        parts = base_name.split("_vs_")
        if len(parts) == 2:
            A, B = parts
        else:
            print(f"[Skip] Invalid column name: {base_name}")
            continue

        # Check if DPO is Assistant A
        dpo_A = "dpo" in A.lower()
        opponent = B if dpo_A else A

        print(f"\nRunning evaluation: {A} vs {B}")
        print(f"→ Output: {output_jsonl.name}")
        print(f"→ dpo_A is A: {dpo_A}")
        print(f"→ Opponent: {opponent}")

        # Run inference + record results
        cloud = generate_responses(
            df=cloud,
            prompt_column=col,
            output_column=output_col,
            output_jsonl=str(output_jsonl),
            inference_fn=inference,
            opponent=opponent,
            dpo_A=dpo_A,
            verdict_column=verdict_col,
        )

        # 🔹 Print verdict counts for THIS vs-pair
        if verdict_col in cloud.columns:
            vc = cloud[verdict_col].value_counts(dropna=True)
            print("\nVerdict counts for", base_name)
            print(vc.to_string())
        else:
            print(f"[Warn] Verdict column not found: {verdict_col}")



Running evaluation: dpo_zero vs vlm_cot
→ Output: dpo_zero_vs_vlm_cot_llm_judge_235.jsonl
→ dpo_A is A: True
→ Opponent: vlm_cot


Generating responses:   0%|          | 1/235 [00:09<35:15,  9.04s/it]


--- Sample Prompt ---
[
  {
    "type": "text",
    "text": "\n[User Question]\nOCR Text:\nimg0:\nTables:\nTable 0:\n+------------------------------+----------------------------------------+---+\n| users                        | q1Cj68vKicgdXt9flxMQkC6qgyf1           | ? |\n|                              | ...                                    |   |\n+------------------------------+----------------------------------------+---+\n| + Add document               | + Start collection                     | + |\n+------------------------------+----------------------------------------+---+\n| LKB3YgolzsNCyQVNfc9ToQrdEq53 | favorites                              |   |\n|                              | >                                      |   |\n+------------------------------+----------------------------------------+---+\n| q1Cj68vKicgdXt9fIxMQkC6qgyf1 |                                        |   |\n+------------------------------+----------------------------------------+---+\n|            

Generating responses:  21%|██▏       | 50/235 [38:10<2:21:13, 45.80s/it]


KeyboardInterrupt: 

# Extract Score 

### Wison_CI 

In [46]:
import math

In [47]:
def wilson_ci(wins, losses, ties=0, z=1.96, tie_as_half=True):
    """95% Wilson CI for win rate. If tie_as_half, use wins+0.5*ties over total; else ignore ties."""
    if tie_as_half:
        n = wins + losses + ties
        if n == 0:
            return (float('nan'), float('nan'), float('nan'))
        phat = (wins + 0.5 * ties) / n
    else:
        n = wins + losses
        if n == 0:
            return (float('nan'), float('nan'), float('nan'))
        phat = wins / n

    denom = 1 + z * z / n
    center = (phat + z * z / (2 * n)) / denom
    half = z * math.sqrt((phat * (1 - phat) + z * z / (4 * n)) / n) / denom
    return phat, max(0.0, center - half), min(1.0, center + half)

### Binom_cdf

In [48]:
def binom_cdf(k, n, p=0.5):
    """Cumulative P(X <= k) for X~Binom(n,p); exact via sum of pmf."""
    comb = math.comb
    return sum(comb(n, i) * (p ** i) * ((1 - p) ** (n - i)) for i in range(k + 1))


In [49]:
def exact_binom_test_two_sided(wins, losses, p=0.5):
    """Two-sided exact binomial p-value on wins vs losses (ignoring ties)."""
    n = wins + losses
    if n == 0:
        return float('nan')
    k = wins
    tail = binom_cdf(min(k, n - k), n, p)
    pval = min(1.0, 2 * tail)
    return pval


### Load and count

In [64]:
import pandas as pd
import json, re
from pathlib import Path
from collections import defaultdict, Counter
import pathlib as path

df=pd.read_excel("../../outputs/scores/llm_judge/cloud/cloud_dpo_vs_llm_ocr_cot_llm_judge.xlsx")

In [65]:
def detect_models_from_cols(columns):
    """
    Find a column like:
      <modelX>_vs_<modelY>_response_verdict
    or related prefixes, then extract modelX, modelY.
    """
    patterns = [
        r"(.+_vs_.+)_response_verdict$",
        r"(.+_vs_.+)_response$",
        r"(.+_vs_.+)_prompt",  # e.g., prompt_column
    ]
    for col in columns:
        for p in patterns:
            m = re.match(p, col)
            if m:
                battle = m.group(1)
                if "_vs_" in battle:
                    left, right = battle.split("_vs_", 1)
                    return left, right, battle
    return None, None, None

left_model, right_model, battle_name = detect_models_from_cols(df.columns)

# Fallback to filename if no column match found
if left_model is None:
    base_name = path.stem.replace("_llm_judge", "")
    left_model, right_model = base_name.split("_vs_") if "_vs_" in base_name else ("ModelA", "ModelB")
    battle_name = f"{left_model}_vs_{right_model}"

# Normalize order so dpo_zero is always first in printing
if "dpo_zero" in left_model:
    dpo_name, llm_name = left_model, right_model
elif "dpo_zero" in right_model:
    dpo_name, llm_name = right_model, left_model
else:
    # If "dpo_zero" string isn't in names, still keep left as "dpo-side"
    dpo_name, llm_name = left_model, right_model

print("Detected battle:", battle_name)
print("Printing as:", f"{dpo_name}_vs_{llm_name}")

Detected battle: dpo_zero_vs_llm_ocr_cot
Printing as: dpo_zero_vs_llm_ocr_cot


In [66]:
required = ["index", "opponent", "dpo_A", "filter_verdict"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["dpo_A"] = df["dpo_A"].fillna(False).astype(bool)

In [67]:
def norm_verdict(v):
    if v is None:
        return "[[C]]"
    s = str(v).strip().upper()
    if s in {"", "NONE", "NULL", "NAN"}:
        return "[[C]]"

    if s in {"[[A]]", "[[B]]", "[[C]]"}:
        return s

    # explicit tie patterns like [[A=B]] / [[A==B]] / B=A
    if re.search(r"(A\s*={1,2}\s*B)|(B\s*={1,2}\s*A)", s):
        return "[[C]]"
    if "TIE" in s or "DRAW" in s:
        return "[[C]]"

    # handles [[A>>B]], [[A>B]], etc.
    m = re.search(r"\[\[\s*([ABC])", s)
    if m:
        return f"[[{m.group(1)}]]"

    if s.startswith("A"): return "[[A]]"
    if s.startswith("B"): return "[[B]]"
    if s.startswith("C"): return "[[C]]"
    return "[[C]]"

df["filter_verdict"] = df["filter_verdict"].apply(norm_verdict)



In [68]:
overall_llm = Counter(w=0, l=0, t=0)    # llm_ocr_cot perspective
overall_dpo0 = Counter(w=0, l=0, t=0)   # dpo_zero perspective

per_opp_dpo0 = defaultdict(lambda: Counter(w=0, l=0, t=0))

def add(counter, w=0, l=0, t=0):
    counter["w"] += w
    counter["l"] += l
    counter["t"] += t

for qid, opp, dpo_A, verdict in df[required].itertuples(index=False, name=None):

    # dpo_A True  => A=dpo_zero, B=llm
    # dpo_A False => A=llm, B=dpo_zero
    llm_is_A = not dpo_A

    if verdict == "[[C]]":
        add(overall_llm, t=1)
        add(overall_dpo0, t=1)
        add(per_opp_dpo0[llm_name], t=1)

    elif verdict == "[[A]]":
        # Slot A won
        if llm_is_A:
            add(overall_llm, w=1)
            add(overall_dpo0, l=1)
            add(per_opp_dpo0[llm_name], l=1)
        else:
            add(overall_llm, l=1)
            add(overall_dpo0, w=1)
            add(per_opp_dpo0[llm_name], w=1)

    elif verdict == "[[B]]":
        # Slot B won
        if llm_is_A:
            add(overall_llm, l=1)
            add(overall_dpo0, w=1)
            add(per_opp_dpo0[llm_name], w=1)
        else:
            add(overall_llm, w=1)
            add(overall_dpo0, l=1)
            add(per_opp_dpo0[llm_name], l=1)


In [69]:
print(
    f"\n{dpo_name}_vs_{llm_name} → "
    f"{dpo_name} wins: {overall_dpo0['w']}, "
    f"{llm_name} wins: {overall_llm['w']}, "
    f"ties: {overall_llm['t']}"
)

winner = (
    dpo_name if overall_dpo0["w"] > overall_dpo0["l"]
    else llm_name if overall_dpo0["w"] < overall_dpo0["l"]
    else "None"
)
print("Winner:", winner)

# Per-opponent (in your data, opponent is constant; force label to llm model)
for opp, cnt in per_opp_dpo0.items():
    print(
        f"{dpo_name}_vs_{opp} → wins: {cnt['w']}, losses: {cnt['l']}, ties: {cnt['t']}"
    )



dpo_zero_vs_llm_ocr_cot → dpo_zero wins: 86, llm_ocr_cot wins: 75, ties: 74
Winner: dpo_zero
dpo_zero_vs_llm_ocr_cot → wins: 86, losses: 75, ties: 74


In [70]:
def format_report(tag, cnt):
    W, L, T = cnt["w"], cnt["l"], cnt["t"]
    wr_half, lo_half, hi_half = wilson_ci(W, L, T, z=1.96, tie_as_half=True)
    wr_ign, lo_ign, hi_ign = wilson_ci(W, L, T, z=1.96, tie_as_half=False)
    pval = exact_binom_test_two_sided(W, L, p=0.5)
    n = W + L + T
    return {
        "system": tag,
        "pairs": n,
        "wins": W,
        "ties": T,
        "losses": L,
        "winrate_tie0.5": round(wr_half, 4),
        "CI95_tie0.5": (round(lo_half, 4), round(hi_half, 4)),
        "winrate_ignore_ties": round(wr_ign, 4),
        "CI95_ignore_ties": (round(lo_ign, 4), round(hi_ign, 4)),
        "binom_p_ignore_ties": round(pval, 6),
    }


In [71]:
# OVERALL summary from dpo_zero perspective
report_rows = [format_report("OVERALL", overall_dpo0)]

# Per-opponent summary (still from dpo_zero perspective)
for opp, cnt in sorted(per_opp_dpo0.items()):
    report_rows.append(format_report(f"{dpo_name} vs {opp}", cnt))


In [74]:
print(report_rows)

[{'system': 'OVERALL', 'pairs': 235, 'wins': 86, 'ties': 74, 'losses': 75, 'winrate_tie0.5': 0.5234, 'CI95_tie0.5': (0.4597, 0.5864), 'winrate_ignore_ties': 0.5342, 'CI95_ignore_ties': (0.4572, 0.6095), 'binom_p_ignore_ties': 0.430725}, {'system': 'dpo_zero vs llm_ocr_cot', 'pairs': 235, 'wins': 86, 'ties': 74, 'losses': 75, 'winrate_tie0.5': 0.5234, 'CI95_tie0.5': (0.4597, 0.5864), 'winrate_ignore_ties': 0.5342, 'CI95_ignore_ties': (0.4572, 0.6095), 'binom_p_ignore_ties': 0.430725}]
